In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
import pickle
from astropy.constants import e, h, c
from scipy.interpolate import interp1d
import pandas as pd
from tcbp.reduction.dataset import get_calib_path

from tcbp.analysis.lsst import LSSTRun

%matplotlib widget

In [ ]:
def plot_Qphotodiodes(run):
    """
    Compute and plot charges ratio between sphere and receiver photodiodes for one run

    run : LSSTRun
        LSSTRun object containing the data for the run
    Inv_electrometers : bool
        If True, the electrometers are inverted (this is the case for the ZTF and Stardice runs)
    """

    ### Get data
    wls = run["set_wl"]
    
    Q_sphere = run['pd_charge_per_burst'][wls > 0.]
    Q_receiver = run['calib_charge_per_burst'][wls > 0.]

    Q_sphere_err = run['pd_charge_per_burst_err'][wls > 0.]
    Q_receiver_err = run['calib_charge_per_burst_err'][wls > 0.]

    ratio = np.abs(Q_receiver / Q_sphere)
    ratio_err = np.abs(ratio) * np.sqrt((Q_sphere_err / Q_sphere)**2 + (Q_receiver_err / Q_receiver)**2)

    ### Plot
    fig, (ax1,ax2,ax3) = plt.subplots(3, sharex="all", figsize=(8,8))

    ax1.errorbar(wls[wls > 0.], Q_sphere, yerr=Q_sphere_err, c='b', marker='.', ls="")
    ax1.set_ylabel('Charges Sphere [C]')
        
    ax2.errorbar(wls[wls > 0.], Q_receiver, yerr=Q_receiver_err, c='r', marker='.', ls="")
    ax2.set_ylabel('Charges Receiver [C]')

    ax3.errorbar(wls[wls > 0.], ratio, yerr=ratio_err, c='indigo', marker='.', ls="")
    ax3.set_ylabel('Ratio Receiver / Sphere [No unit]')
    ax3.set_xlabel('wavelengths [nm]')

    fig.tight_layout()
    plt.show()

    return ratio, ratio_err

## Load and extract a specific run, create a npy file with the data

In [ ]:
main_repo = '/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/data/'

#Inv_electrometers = False
#data_dir = main_repo + '20260627_213502_optical_bench_pinhole5mm_slit20/'
#data_dir = main_repo + '20260628_193214_optical_bench_pinhole5mm_slit20/'
#data_dir = main_repo + '20260628_214053_optical_bench_pinhole1mm_slit20/'
#data_dir = main_repo + '20260629_004614_optical_bench_pinhole200um_slit20/'
#data_dir = main_repo + '20260629_134742_optical_bench_pinhole5mm_slit20/'
#data_dir = main_repo + '20260629_152137_optical_bench_pinhole5mm_slit20/'
#data_dir = main_repo + '20260630_175411_optical_bench_pinhole5mm_slit20/'
#data_dir = main_repo + '20260630_195101_optical_bench_pinhole5mm_slit20/'
#data_dir = main_repo + '20260701_000936_optical_bench_pinhole5mm_slit20/'
#data_dir = main_repo + '20260708_002717_optical_bench_pinhole5mm_8D057_slit20/'


Inv_electrometers = True
#data_dir = main_repo + '20260629_170826_optical_bench_pinhole5mm_ZTFlike_slit20/'
#data_dir = main_repo + '20260629_194844_optical_bench_pinhole200um_ZTFlike_slit20/'
#data_dir = main_repo + '20260629_213351_optical_bench_pinhole5mm_SDlike_slit20/'
#data_dir = main_repo + '20260701_140913_optical_bench_pinhole5mm_ZTFlike_slit20/'
data_dir = main_repo + '20260708_013122_optical_bench_pinhole5mm_8D057_SDLike_slit20/'

In [ ]:
if Inv_electrometers is True:
    print("Electrometers are inverted for this run")
    pd_extname="KEITHLEY" # Sphere
    sc_extname="KEYSIGHT" # NIST
else :
    print("Electrometers are not inverted for this run")
    pd_extname="KEYSIGHT" # Sphere
    sc_extname="KEITHLEY" # NIST

## Do the reduction
run = LSSTRun(data_dir,
              nbursts=1, spectro_calib_path="2026_Auxtel/20260429", 
              spectro_temp_calib=False,
              pd_extname=pd_extname, sc_extname=sc_extname,
              pins={"KEYSIGHT":1, "KEITHLEY":4, "SHUTTER":2},
              ticks_per_shutter_event=1,
              use_digital_analyzer=True)
run.load()
run.extraction()
run.plot_summary()

np.save(data_dir + "run.npy", run.data)

### Ratio Receiver / Sphere 

In [ ]:
run = np.load(data_dir + "run.npy", allow_pickle=True)
wls_run = run["set_wl"][run["set_wl"] > 300.]
ratio, ratio_err = plot_Qphotodiodes(run)

print(np.mean(np.abs(ratio_err) / ratio))


## Thierry and Kelian data

In [ ]:
### Thierry's data
tcbp = np.load(get_calib_path("cbp_laser_response_dr5.npy"), allow_pickle=True)
tcbp_func = interp1d(tcbp['cbp_wl'], tcbp['cbp_tr'], bounds_error=False, fill_value="extrapolate")
tcbp_err_func = interp1d(tcbp['cbp_wl'], tcbp['cbp_tr_err'], bounds_error=False, fill_value="extrapolate")

### Kelian's data
solarcell = np.loadtxt(get_calib_path("SC_QE_NewCell_20220401_LongQECurve-2_reduced_filtered.txt"), skiprows=1, delimiter=",")
pd_df_enhanced = pd.read_feather(get_calib_path("tCBP_v0_pd_df_fit.feather"))
sc_df_enhanced = pd.read_feather(get_calib_path("tCBP_v0_sc_df_fit.feather"))

sigma_A_SC = sc_df_enhanced['sigma_amplitude']
A_SC = sc_df_enhanced['amplitude']

sigma_A_PD = pd_df_enhanced['sigma_amplitude']
A_PD = pd_df_enhanced['amplitude']

T_CBP = A_SC/A_PD/np.interp(sc_df_enhanced['wl'], solarcell[:,0], solarcell[:,1])
T_CBP_wl = sc_df_enhanced['wl']

sigma_T_CBP = T_CBP * np.sqrt((sigma_A_SC/A_SC)**2 + (sigma_A_PD/A_PD)**2)


## Lens transmission and NIST photodiode efficiency

In [ ]:
wls_interp = np.arange(300, 1150, 4)


### Elana lens measurement
data_dir2 = '/home/lmousset/Projets_Recherche/LSST/Mesure_transmission_tcbp/'
with open(data_dir2 + 'Interp_lens_transmission.pkl', 'rb') as file:
   interpolator_lens = pickle.load(file)

#transmission_lens_interp = interpolator_lens(wls_run)


### New lens transmission data
new_lens_transmission = np.load('/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/lens_transmission_data_new.npy', allow_pickle=True)
new_interpolator_lens = interp1d(new_lens_transmission[0, :], new_lens_transmission[1, :], kind='cubic', fill_value='extrapolate')
new_interpolator_lens_err = interp1d(new_lens_transmission[0, :], new_lens_transmission[2, :], kind='linear', fill_value='extrapolate')

transmission_lens_interp = new_interpolator_lens(wls_interp)
transmission_lens_interp_err = new_interpolator_lens_err(wls_interp)

# NIST photodiode calibration data
NIST_e_per_photon = np.load(data_dir2 + 'NIST_e_per_photon.npy', allow_pickle=True)
interpolator_pd_NIST_e_per_photon = interp1d(NIST_e_per_photon[0, :], NIST_e_per_photon[1, :], kind='cubic', fill_value='extrapolate')
interpolator_pd_NIST_e_per_photon_err = interp1d(NIST_e_per_photon[0, :], NIST_e_per_photon[2, :], kind='linear', fill_value='extrapolate')

pd_NIST_interp_e_per_photon = interpolator_pd_NIST_e_per_photon(wls_interp)
pd_NIST_interp_e_per_photon_err = interpolator_pd_NIST_e_per_photon_err(wls_interp)

# Marc photodiode data
pd8D057_Qeff = np.load(data_dir2 + 'photodiode_Qeff_8D057.npy', allow_pickle=True)
interpolator_pd8D057_Qeff = interp1d(pd8D057_Qeff[0, :], pd8D057_Qeff[1, :], kind='cubic', fill_value='extrapolate')
interpolator_pd8D057_Qeff_err = interp1d(pd8D057_Qeff[0, :], pd8D057_Qeff[2, :], kind='linear', fill_value='extrapolate')

pd8D057_interp_Qeff = interpolator_pd8D057_Qeff(wls_interp)
pd8D057_interp_Qeff_err = interpolator_pd8D057_Qeff_err(wls_interp)


In [ ]:
plt.figure(figsize=(10, 6))
#plt.errorbar(wls_interp, transmission_lens_interp, yerr=transmission_lens_interp_err, label='Lens transmission', c='k', marker='.', ls="")
plt.errorbar(wls_interp, pd_NIST_interp_e_per_photon, yerr=pd_NIST_interp_e_per_photon_err, label='NIST photodiode calibration', c='r', marker='.', ls="")
plt.errorbar(wls_interp, pd8D057_interp_Qeff, yerr=pd8D057_interp_Qeff_err, label='Marc photodiode 8D057', c='g', marker='.', ls="")
plt.legend()
plt.xlabel('Wavelength [nm]')


# Get several runs

In [ ]:
data_dir_list = [main_repo + '20260627_213502_optical_bench_pinhole5mm_slit20/',        # Run 0 Tout premier run (colle avec les runs 4-5-6)
                 #main_repo + '20260628_193214_optical_bench_pinhole5mm_slit20/',        # Run 1 Papier écrasé
                 #main_repo + '20260629_134742_optical_bench_pinhole5mm_slit20/',        # Run 2 Papier écrasé (sampling max)
                 #main_repo + '20260629_152137_optical_bench_pinhole5mm_slit20/',        # Run 3 Papier écrasé (sampling max)
                 main_repo + '20260630_195101_optical_bench_pinhole5mm_slit20/',        # Run 4 Après avoir refait le foyer sur le vert
                 main_repo + '20260701_000936_optical_bench_pinhole5mm_slit20/',        # Run 5 Idem que Run 4
                 main_repo + '20260630_175411_optical_bench_pinhole5mm_slit20/',        # Run 6 Foyer sur le bleu 
                 #main_repo + '20260628_214053_optical_bench_pinhole1mm_slit20/',        # Run 7 Trou de 1mm
                 #main_repo + '20260629_004614_optical_bench_pinhole200um_slit20/',      # Run 8 Trou de 200µm, bruité mais ok
                 #main_repo + '20260629_170826_optical_bench_pinhole5mm_ZTFlike_slit20/',# Run 9 ZTF-like
                 #main_repo + '20260629_194844_optical_bench_pinhole200um_ZTFlike_slit20/', # Run 10 ZTF-like 200µm
                 #main_repo + '20260629_213351_optical_bench_pinhole5mm_SDlike_slit20/',  # Run 11 StarDice-like
                 #main_repo + '20260701_140913_optical_bench_pinhole5mm_ZTFlike_slit20/',  # Run 12 ZTF-like
                 main_repo + '20260708_002717_optical_bench_pinhole5mm_8D057_slit20/',
                 main_repo + '20260708_013122_optical_bench_pinhole5mm_8D057_SDLike_slit20/'
                 ]

nruns = len(data_dir_list)

all_ratio = []
all_ratio_err = []
all_wls_run = []
for i, data_dir in enumerate(data_dir_list):
    print(i)

    run = np.load(data_dir + "run.npy", allow_pickle=True)

    wls_run = run["set_wl"][run["set_wl"] > 0.]
    print(wls_run)
    ratio, ratio_err = plot_Qphotodiodes(run)

    all_wls_run.append(wls_run)
    all_ratio.append(ratio)
    all_ratio_err.append(ratio_err)

    print(np.mean(np.abs(ratio_err) / ratio))

In [ ]:
# Sort by wavelength
all_wls_run_sorted = []
for i in range(nruns):
    sort_indices = np.argsort(all_wls_run[i])
    all_wls_run_sorted.append(all_wls_run[i][sort_indices])
    all_ratio[i] = all_ratio[i][sort_indices]
    all_ratio_err[i] = all_ratio_err[i][sort_indices]

In [ ]:
## Plot all ratios
plt.figure(figsize=(10,5))
for i in range(nruns):
    print(i)
    #norm = all_ratio[i][all_wls_run_sorted[i]==769.996][0]
    norm = 1.#all_ratio[i][all_wls_run_sorted[i]==500.][0]
    print(norm)
    plt.errorbar(all_wls_run_sorted[i], all_ratio[i] / norm, yerr=np.abs(all_ratio_err[i] / norm), marker='.', ls="", label=f'Run {i}')
plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere')
plt.grid(True)
#plt.ylim(0, 2.5)
plt.legend()

In [ ]:
# Rapport entre 2 scans
irun1 = 2
irun2 = 5

commun_wls = np.intersect1d(all_wls_run_sorted[irun1], all_wls_run_sorted[irun2])
print(f"Commun wavelengths between run {irun1} and run {irun2}: {commun_wls}")

# Utiliser searchsorted pour maintenir l'ordre correct des longueurs d'onde
indices1 = np.searchsorted(all_wls_run_sorted[irun1], commun_wls)
indices2 = np.searchsorted(all_wls_run_sorted[irun2], commun_wls)

commun_ratio1 = all_ratio[irun1][indices1]
commun_ratio2 = all_ratio[irun2][indices2]

commun_ratio_err1 = all_ratio_err[irun1][indices1]
commun_ratio_err2 = all_ratio_err[irun2][indices2]

rapport = commun_ratio1 / commun_ratio2
rapport_err = np.sqrt((commun_ratio_err1 / commun_ratio1)**2 + (commun_ratio_err2 / commun_ratio2)**2) * rapport


plt.figure(figsize=(10,5))
plt.errorbar(commun_wls, rapport, yerr=rapport_err, c='orange', marker='.', ls="")
#plt.ylim(0.99, 1.01)
plt.ylabel('ratio entre deux runs')

In [ ]:
### Correction factor
all_ratio_corrected = []
all_ratio_corrected_err = []
for i in range(nruns):
    transmission_lens_interp = new_interpolator_lens(all_wls_run_sorted[i]) # New lens measurement
    transmission_lens_interp_err = new_interpolator_lens_err(all_wls_run_sorted[i]) # New lens measurement
    #transmission_lens_interp = interpolator_lens(all_wls_run_sorted[i]) # Elana measurement

    if i in [4, 5]:
        print(f"Run {i} is using 8D057 photodiode")
        pd_interp_e_per_photon = interpolator_pd8D057_Qeff(all_wls_run_sorted[i])
        pd_interp_e_per_photon_err = interpolator_pd8D057_Qeff_err(all_wls_run_sorted[i])
    
    else :
        print(f"Run {i} is using NIST photodiode")
        pd_interp_e_per_photon = interpolator_pd_NIST_e_per_photon(all_wls_run_sorted[i])
        pd_interp_e_per_photon_err = interpolator_pd_NIST_e_per_photon_err(all_wls_run_sorted[i])

    correction_factor = 1 / (pd_interp_e_per_photon * transmission_lens_interp * e.value) 
    ratio_corrected = all_ratio[i] * correction_factor

    ### Erreurs
    ratio_corrected_err = np.sqrt((all_ratio_err[i]/all_ratio[i])**2 + 
                                  (pd_interp_e_per_photon_err / pd_interp_e_per_photon)**2 + 
                                  (transmission_lens_interp_err / transmission_lens_interp)**2) * ratio_corrected
    
    all_ratio_corrected.append(ratio_corrected)
    all_ratio_corrected_err.append(ratio_corrected_err)

In [ ]:
plt.figure(figsize=(10,5))
for i in range(nruns):
    #norm = all_ratio_corrected[i][all_wls_run_sorted[i]==500.][0]
    #print(norm)
    plt.errorbar(all_wls_run_sorted[i], all_ratio_corrected[i], yerr=np.abs(all_ratio_corrected_err[i]), marker='.', ls="-", label=f'Run {i}')
plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere corrected')
plt.grid(True)
plt.legend()

In [ ]:
# Rapport entre 2 scans après avoir appliqué le facteur de correction
irun1 = 2
irun2 = 5

commun_wls = np.intersect1d(all_wls_run_sorted[irun1], all_wls_run_sorted[irun2])
print(f"Commun wavelengths between run {irun1} and run {irun2}: {commun_wls}")

# Utiliser searchsorted pour maintenir l'ordre correct des longueurs d'onde
indices1 = np.searchsorted(all_wls_run_sorted[irun1], commun_wls)
indices2 = np.searchsorted(all_wls_run_sorted[irun2], commun_wls)

commun_ratio1 = all_ratio_corrected[irun1][indices1]
commun_ratio2 = all_ratio_corrected[irun2][indices2]

commun_ratio_err1 = all_ratio_corrected_err[irun1][indices1]
commun_ratio_err2 = all_ratio_corrected_err[irun2][indices2]

rapport = commun_ratio1 / commun_ratio2
rapport_err = np.sqrt((commun_ratio_err1 / commun_ratio1)**2 + (commun_ratio_err2 / commun_ratio2)**2) * rapport


plt.figure(figsize=(10,5))
plt.errorbar(commun_wls, rapport, yerr=np.abs(rapport_err), c='orange', marker='.', ls="")
#plt.ylim(0.9, 1.05)
plt.ylabel('ratio entre deux runs')

In [ ]:
# Save a run
irun = 2
my_run = np.zeros((3, len(all_wls_run_sorted[irun])))
my_run[0,:] = all_wls_run_sorted[irun]
my_run[1,:] = all_ratio_corrected[irun]
my_run[2,:] = all_ratio_corrected_err[irun]

# Transformer ce tableau en record array
np.save(data_dir_list[irun] + "run_corrected_new_lens_transmission.npy", np.core.records.fromarrays(my_run, names='cbp_wl, cbp_tr, cbp_tr_err', formats='f8,f8,f8'))
